# 🚗 Dashcam AI Pipeline
### YOLO Object & Sign Detection + Lane Detection (SCNN) + BEV Distance Estimation
**Team:** Jeevan (EC23B1016) · Arulmozhi (EC23B1135) · Sudhiksha (ME23B2006)  
**Course:** Multimedia Processing and Analysis (EC5110) · IIITDM Kancheepuram  
**Guide:** Dr. Mukku Nisanth Kartheek

> ✅ **Before running:** Go to `Runtime > Change Runtime Type > T4 GPU`
>
> 📦 **What this notebook produces (all auto-downloaded at Cell 10):**
> - `output_processed.mp4` — fully annotated output video
> - `latency_report.png` — latency chart
> - `latency_data.csv` — latency numbers (for report table)
> - `sample_frames/` — 6 sample output frames as PNG (for report figures)
> - `input_frames/` — 4 input frames as PNG (for report comparison)
> - `bev_roi_frame.png` — BEV ROI region visualization
> - `lane_only_frame.png` — lane detection only frame
> - `yolo_only_frame.png` — YOLO detection only frame
> - `pipeline_summary.txt` — plain-text summary of run stats


## Cell 1 — Install Dependencies

In [ ]:
!pip install ultralytics -q
!pip install torch torchvision -q
!pip install opencv-python-headless -q
!pip install numpy matplotlib tqdm pandas -q
!pip install gdown -q

import os
if not os.path.exists('/content/SCNN_Pytorch'):
    !git clone https://github.com/harryhan618/SCNN_Pytorch.git -q
    %cd SCNN_Pytorch
    !pip install -r requirements.txt -q
    %cd ..
else:
    print("SCNN_Pytorch already cloned")

!mkdir -p /content/SCNN_Pytorch/experiments/exp0
!mkdir -p /content/SCNN_Pytorch/experiments/vgg_SCNN_DULR_w9

# Output directories for report assets
!mkdir -p /content/report_assets/sample_frames
!mkdir -p /content/report_assets/input_frames

print("\n✅ All dependencies installed!")
print("⚡ GPU Status:", end=" ")
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "No GPU found — switch Runtime to T4!"


## Cell 2 — Download LaneNet (SCNN) Pretrained Weights
> **TuSimple trained model** — 94.16% accuracy. Download the `.pth` file from the [SCNN_Pytorch README](https://github.com/harryhan618/SCNN_Pytorch/blob/master/README.md) (Google Drive link) and upload it below.

In [ ]:
from google.colab import files
import os, shutil

WEIGHT_PATH = '/content/SCNN_Pytorch/experiments/exp0/exp0.pth'

if os.path.exists(WEIGHT_PATH):
    print(f"✅ Weights already present at {WEIGHT_PATH}")
else:
    print("📥 Upload your SCNN TuSimple weights file (exp0.pth)...")
    print(" Download from: https://github.com/harryhan618/SCNN_Pytorch (README > TuSimple model > here)")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, WEIGHT_PATH)
    print(f"✅ Weights saved to {WEIGHT_PATH}")


## Cell 3 — Load YOLO (Auto-Download)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")
print("✅ YOLOv8n loaded —", sum(p.numel() for p in yolo_model.model.parameters()), "parameters")
print(" Classes available:", len(yolo_model.names), "| e.g.:", list(yolo_model.names.values())[:8])


## Cell 4 — Load LaneNet (SCNN)

In [ ]:
import sys, torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import cv2

sys.path.insert(0, '/content/SCNN_Pytorch')
LANENET_AVAILABLE = False

try:
    from model import SCNN
    lane_model = SCNN(input_size=(800, 288), pretrained=False)
    WEIGHT_PATH = '/content/SCNN_Pytorch/experiments/exp0/exp0.pth'
    checkpoint = torch.load(WEIGHT_PATH, map_location='cpu')
    state = checkpoint.get('state_dict', checkpoint)
    lane_model.load_state_dict(state)
    lane_model.eval()
    if torch.cuda.is_available():
        lane_model = lane_model.cuda()
        print("✅ LaneNet (SCNN) loaded on GPU")
    else:
        print("✅ LaneNet (SCNN) loaded on CPU")
    LANENET_AVAILABLE = True
except Exception as e:
    print(f"⚠️ LaneNet not loaded: {e}")
    print(" → Falling back to OpenCV Canny+Hough lane detection")
    LANENET_AVAILABLE = False

SCNN_MEAN = (0.485, 0.456, 0.406)
SCNN_STD  = (0.229, 0.224, 0.225)
scnn_transform = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize(mean=SCNN_MEAN, std=SCNN_STD)
])

print(f"\nLaneNet available: {LANENET_AVAILABLE}")


## Cell 5 — All Helper Functions (BEV + Lane + Report Utilities)

In [ ]:
import math, time

# ── BEV Homography Distance Estimation ─────────────────────────────────────
def estimate_bev_distance(frame, boxes):
    h, w = frame.shape[:2]
    src = np.float32([
        [w * 0.15, h], [w * 0.85, h],
        [w * 0.62, h * 0.62], [w * 0.38, h * 0.62]
    ])
    dst = np.float32([
        [w * 0.20, h], [w * 0.80, h],
        [w * 0.80, 0], [w * 0.20, 0]
    ])
    H = cv2.getPerspectiveTransform(src, dst)
    PIXELS_PER_METER = (w * 0.60) / 3.5
    ego_pt  = np.float32([[[w / 2, h]]])
    ego_bev = cv2.perspectiveTransform(ego_pt, H)
    distances = []
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        bot_center = np.float32([[[(x1 + x2) / 2, y2]]])
        bev_pt = cv2.perspectiveTransform(bot_center, H)
        px_dist = math.hypot(
            bev_pt[0, 0, 0] - ego_bev[0, 0, 0],
            bev_pt[0, 0, 1] - ego_bev[0, 0, 1]
        )
        distances.append(round(px_dist / PIXELS_PER_METER, 1))
    return distances


# ── OpenCV Fallback Lane Detection ─────────────────────────────────────────
def detect_lanes_cv(frame):
    h, w = frame.shape[:2]
    mask = np.zeros_like(frame[:, :, 0])
    poly = np.array([[
        (int(w * 0.08), h), (int(w * 0.44), int(h * 0.60)),
        (int(w * 0.56), int(h * 0.60)), (int(w * 0.92), h)
    ]], dtype=np.int32)
    cv2.fillPoly(mask, poly, 255)
    gray   = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur   = cv2.GaussianBlur(gray, (5, 5), 0)
    edges  = cv2.Canny(blur, 50, 150)
    masked = cv2.bitwise_and(edges, edges, mask=mask)
    lines  = cv2.HoughLinesP(masked, 1, np.pi / 180, threshold=30,
                             minLineLength=50, maxLineGap=120)
    overlay = frame.copy()
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(overlay, (x1, y1), (x2, y2), (0, 165, 255), 3)
    return cv2.addWeighted(frame, 0.80, overlay, 0.50, 0)


# ── SCNN Lane Detection ─────────────────────────────────────────────────────
def detect_lanes_scnn(frame):
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inp = scnn_transform(img_pil).unsqueeze(0)
    if torch.cuda.is_available():
        inp = inp.cuda()
    with torch.no_grad():
        seg_pred, exist_pred = lane_model(inp)
    seg = seg_pred.squeeze().cpu().numpy()
    lane_map = (np.argmax(seg, axis=0) * 60).astype(np.uint8)
    lane_resized = cv2.resize(lane_map, (frame.shape[1], frame.shape[0]))
    colored = cv2.applyColorMap(lane_resized, cv2.COLORMAP_JET)
    colored[lane_resized == 0] = 0
    return cv2.addWeighted(frame, 0.80, colored, 0.55, 0)


def detect_lanes(frame):
    if LANENET_AVAILABLE:
        try:
            return detect_lanes_scnn(frame)
        except Exception:
            pass
    return detect_lanes_cv(frame)


# ── BEV Minimap ─────────────────────────────────────────────────────────────
def draw_bev_minimap(frame, boxes, distances):
    h, w = frame.shape[:2]
    map_h, map_w = 160, 120
    bev_map = np.zeros((map_h, map_w, 3), dtype=np.uint8)
    bev_map[:] = (30, 30, 30)
    ego_x, ego_y = map_w // 2, map_h - 20
    cv2.rectangle(bev_map, (ego_x - 8, ego_y - 14), (ego_x + 8, ego_y + 14), (0, 200, 0), -1)
    for i, box in enumerate(boxes):
        if i >= len(distances): break
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        dist = distances[i]
        bev_obj_y = int(ego_y - min(dist, 50) / 50 * (map_h - 40))
        bev_obj_x = ego_x + int(((x1 + x2) / 2 - w / 2) / w * map_w * 0.8)
        bev_obj_x = max(10, min(map_w - 10, bev_obj_x))
        cv2.rectangle(bev_map, (bev_obj_x - 6, bev_obj_y - 10),
                      (bev_obj_x + 6, bev_obj_y + 10), (0, 165, 255), -1)
        cv2.putText(bev_map, f"{dist}m", (bev_obj_x - 10, bev_obj_y - 13),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255, 255, 255), 1)
    cv2.line(bev_map, (int(map_w*0.35), map_h), (int(map_w*0.45), 0), (0, 165, 255), 1)
    cv2.line(bev_map, (int(map_w*0.65), map_h), (int(map_w*0.55), 0), (0, 165, 255), 1)
    cv2.putText(bev_map, "BEV MAP", (5, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (200, 200, 200), 1)
    frame[h - map_h - 10 : h - 10, w - map_w - 10 : w - 10] = bev_map
    return frame


# ── Report Utility: Save BEV ROI Visualisation ─────────────────────────────
def save_bev_roi_visual(frame, save_path):
    """Draws the BEV source trapezoid ROI on a copy of the frame and saves it."""
    h, w = frame.shape[:2]
    vis = frame.copy()
    src_pts = np.array([
        [int(w * 0.15), h], [int(w * 0.85), h],
        [int(w * 0.62), int(h * 0.62)], [int(w * 0.38), int(h * 0.62)]
    ], dtype=np.int32)
    cv2.polylines(vis, [src_pts], isClosed=True, color=(0, 255, 255), thickness=3)
    for pt in src_pts:
        cv2.circle(vis, tuple(pt), 7, (0, 0, 255), -1)
    cv2.putText(vis, "BEV Source ROI", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    cv2.imwrite(save_path, vis)
    print(f"  Saved: {save_path}")


print("✅ All helper functions ready!")


## Cell 6 — Upload Dashcam Video

In [ ]:
from google.colab import files
import shutil, os

print("📁 Upload your dashcam .mp4 video file now...")
uploaded = files.upload()

video_filename = list(uploaded.keys())[0]
INPUT_VIDEO  = f"/content/{video_filename}"
OUTPUT_VIDEO = "/content/output_processed.mp4"
shutil.move(video_filename, INPUT_VIDEO)

cap = cv2.VideoCapture(INPUT_VIDEO)
print(f"\n✅ Video uploaded: {INPUT_VIDEO}")
print(f"   Resolution : {int(cap.get(3))}×{int(cap.get(4))}")
print(f"   FPS        : {cap.get(cv2.CAP_PROP_FPS):.1f}")
print(f"   Frames     : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
print(f"   Duration   : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT)/cap.get(cv2.CAP_PROP_FPS)):.0f} seconds")
cap.release()


## Cell 7 — Process Video (YOLO + LaneNet + BEV)

In [ ]:
from tqdm.notebook import tqdm
import time, cv2, numpy as np

cap    = cv2.VideoCapture(INPUT_VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps, (width, height)
)

latencies   = {"yolo": [], "lane": [], "bev": [], "total": []}
frame_count = 0

# For report: capture specific frames
SAVE_INPUT_AT   = [1, int(total*0.25), int(total*0.50), int(total*0.75)]
SAVE_OUTPUT_AT  = [1, int(total*0.15), int(total*0.30), int(total*0.50), int(total*0.70), int(total*0.90)]
SAVE_BEV_AT     = int(total * 0.10)
SAVE_LANE_AT    = int(total * 0.20)
SAVE_YOLO_AT    = int(total * 0.05)
bev_roi_saved   = False

print(f"🎬 Processing {total} frames | {fps:.1f} FPS | {width}×{height}")
print(f"   LaneNet Mode: {'SCNN (TuSimple)' if LANENET_AVAILABLE else 'OpenCV Fallback'}")
print("─" * 60)

for _ in tqdm(range(total), desc="🚗 Processing dashcam"):
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    t_total = time.perf_counter()

    # Save raw input frame for report
    if frame_count in SAVE_INPUT_AT:
        idx = SAVE_INPUT_AT.index(frame_count) + 1
        cv2.imwrite(f"/content/report_assets/input_frames/input_frame_{idx}.png", frame)

    # ── 1. YOLO ────────────────────────────────────────────────────────
    t0      = time.perf_counter()
    results = yolo_model(frame, conf=0.40, verbose=False)[0]
    boxes   = results.boxes
    frame   = results.plot(line_width=2, font_size=10)
    latencies["yolo"].append(time.perf_counter() - t0)

    # Save YOLO-only frame for report
    if frame_count == SAVE_YOLO_AT:
        cv2.imwrite("/content/report_assets/yolo_only_frame.png", frame)

    # ── 2. Lane Detection ──────────────────────────────────────────────
    t0    = time.perf_counter()
    frame = detect_lanes(frame)
    latencies["lane"].append(time.perf_counter() - t0)

    # Save lane-only frame for report
    if frame_count == SAVE_LANE_AT:
        cv2.imwrite("/content/report_assets/lane_only_frame.png", frame)

    # ── 3. BEV Distance Estimation ─────────────────────────────────────
    t0        = time.perf_counter()
    distances = estimate_bev_distance(frame, boxes)
    for i, box in enumerate(boxes):
        if i >= len(distances): break
        x1 = int(box.xyxy[0][0])
        y1 = int(box.xyxy[0][1])
        cv2.putText(frame, f"{distances[i]}m",
                    (x1, max(y1 - 12, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (0, 165, 255), 2, cv2.LINE_AA)
    latencies["bev"].append(time.perf_counter() - t0)

    # Save BEV ROI visualisation once
    if frame_count == SAVE_BEV_AT and not bev_roi_saved:
        raw_frame_for_bev = cv2.imread(f"/content/report_assets/input_frames/input_frame_1.png")
        if raw_frame_for_bev is not None:
            save_bev_roi_visual(raw_frame_for_bev, "/content/report_assets/bev_roi_frame.png")
        bev_roi_saved = True

    # ── 4. BEV Minimap ─────────────────────────────────────────────────
    frame = draw_bev_minimap(frame, boxes, distances)

    # ── 5. Latency HUD ─────────────────────────────────────────────────
    total_ms = (time.perf_counter() - t_total) * 1000
    latencies["total"].append(total_ms / 1000)
    eff_fps  = 1000 / total_ms

    hud = (f"Frame {frame_count}/{total} | "
           f"Total:{total_ms:.0f}ms | "
           f"YOLO:{latencies['yolo'][-1]*1000:.0f}ms | "
           f"Lane:{latencies['lane'][-1]*1000:.0f}ms | "
           f"BEV:{latencies['bev'][-1]*1000:.0f}ms | "
           f"FPS:{eff_fps:.1f}")
    cv2.rectangle(frame, (0, height - 26), (width, height), (0, 0, 0), -1)
    cv2.putText(frame, hud, (8, height - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.40,
                (0, 165, 255), 1, cv2.LINE_AA)

    # ── 6. Lane Method Label ────────────────────────────────────────────
    method = "LaneNet SCNN" if LANENET_AVAILABLE else "OpenCV Hough"
    cv2.putText(frame, f"Lane: {method}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (255, 255, 0), 2, cv2.LINE_AA)

    # Save annotated output frames for report
    if frame_count in SAVE_OUTPUT_AT:
        idx = SAVE_OUTPUT_AT.index(frame_count) + 1
        cv2.imwrite(f"/content/report_assets/sample_frames/output_frame_{idx}.png", frame)

    out.write(frame)

cap.release()
out.release()
print(f"\n✅ Done! {frame_count} frames processed.")
print(f"📁 Output video: {OUTPUT_VIDEO}")
print(f"🖼️  Sample frames saved to /content/report_assets/")


## Cell 8 — Latency Report + Chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def ms(arr): return np.array(arr) * 1000

print("=" * 60)
print(" LATENCY REPORT (per frame, milliseconds)")
print("=" * 60)
print(f" {'Module':<22} {'Avg':>8} {'Min':>8} {'Max':>8} {'Std':>8}")
print("-" * 60)
for name, arr in latencies.items():
    if not arr: continue
    a = ms(arr)
    print(f" {name:<22} {np.mean(a):>7.1f}ms {np.min(a):>7.1f}ms "
          f"{np.max(a):>7.1f}ms {np.std(a):>7.1f}ms")
print("-" * 60)
avg_fps = 1000 / np.mean(ms(latencies["total"]))
print(f"\n  ⚡ Effective Processing Speed : {avg_fps:.1f} FPS")
print(f"  🎞️  Total frames processed    : {len(latencies['total'])}")
print(f"  📹 Lane detection method      : {'SCNN (TuSimple)' if LANENET_AVAILABLE else 'OpenCV Hough'}")
print("=" * 60)

# ── Plot latency timeline ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
fig.suptitle("Per-Frame Latency Over Time", fontsize=14, fontweight='bold')
colors = ['#FF6B00', '#00BFFF', '#00E676', '#FF4444']
names  = ['yolo', 'lane', 'bev', 'total']
labels = ['YOLO', 'LaneNet', 'BEV Distance', 'Total Pipeline']

for ax, name, label, color in zip(axes.flat, names, labels, colors):
    data = ms(latencies[name])
    ax.plot(data, color=color, linewidth=0.8, alpha=0.8)
    ax.axhline(np.mean(data), color='white', linewidth=1.5,
               linestyle='--', label=f"Avg: {np.mean(data):.1f}ms")
    ax.set_title(label, color='white', fontsize=11)
    ax.set_xlabel("Frame", fontsize=9)
    ax.set_ylabel("ms", fontsize=9)
    ax.legend(fontsize=8)
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='white')

fig.patch.set_facecolor('#13131f')
plt.tight_layout()
plt.savefig("/content/latency_report.png", dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("📊 Latency chart saved: /content/latency_report.png")


## Cell 9 — Save Latency CSV + Pipeline Summary (for Report)

In [ ]:
import pandas as pd
import numpy as np
import os

def ms(arr): return np.array(arr) * 1000

# ── 1. Latency CSV ─────────────────────────────────────────────────────────
rows = []
for name, arr in latencies.items():
    if not arr: continue
    a = ms(arr)
    rows.append({
        "Module":       name.upper(),
        "Avg (ms)":     round(float(np.mean(a)), 2),
        "Min (ms)":     round(float(np.min(a)),  2),
        "Max (ms)":     round(float(np.max(a)),  2),
        "Std (ms)":     round(float(np.std(a)),  2),
        "Median (ms)":  round(float(np.median(a)), 2),
    })

df_lat = pd.DataFrame(rows)
df_lat.to_csv("/content/latency_data.csv", index=False)
print("✅ Latency CSV saved: /content/latency_data.csv")
print(df_lat.to_string(index=False))

# ── 2. Full per-frame latency CSV ──────────────────────────────────────────
cap_check = cv2.VideoCapture(INPUT_VIDEO)
vid_fps   = cap_check.get(cv2.CAP_PROP_FPS)
cap_check.release()

n = len(latencies['total'])
df_frames = pd.DataFrame({
    "Frame":    list(range(1, n + 1)),
    "YOLO_ms":  [round(x * 1000, 2) for x in latencies['yolo']],
    "Lane_ms":  [round(x * 1000, 2) for x in latencies['lane']],
    "BEV_ms":   [round(x * 1000, 2) for x in latencies['bev']],
    "Total_ms": [round(x * 1000, 2) for x in latencies['total']],
    "FPS":      [round(1000 / (x * 1000), 2) for x in latencies['total']],
})
df_frames.to_csv("/content/latency_per_frame.csv", index=False)
print("\n✅ Per-frame latency CSV saved: /content/latency_per_frame.csv")

# ── 3. Pipeline Summary Text (paste into report) ───────────────────────────
cap_info = cv2.VideoCapture(INPUT_VIDEO)
in_w  = int(cap_info.get(cv2.CAP_PROP_FRAME_WIDTH))
in_h  = int(cap_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
in_fps = cap_info.get(cv2.CAP_PROP_FPS)
in_frames = int(cap_info.get(cv2.CAP_PROP_FRAME_COUNT))
cap_info.release()

avg_fps_val = 1000 / float(np.mean(ms(latencies['total'])))

summary = f"""==========================================================
PIPELINE RUN SUMMARY
IIITDM Kancheepuram · EC5110 · Multimedia Project
Team: Jeevan EC23B1016 | Arulmozhi EC23B1135 | Sudhiksha ME23B2006
==========================================================
Input Video Resolution  : {in_w}x{in_h}
Input Video FPS         : {in_fps:.1f}
Total Frames Processed  : {in_frames}
Lane Detection Method   : {'SCNN (TuSimple pretrained)' if LANENET_AVAILABLE else 'OpenCV Canny + Hough (fallback)'}
----------------------------------------------------------
LATENCY SUMMARY
Module         Avg       Min       Max       Std
"""
for name, arr in latencies.items():
    if not arr: continue
    a = ms(arr)
    summary += f"{name.upper():<15}{np.mean(a):>7.1f}ms  {np.min(a):>7.1f}ms  {np.max(a):>7.1f}ms  {np.std(a):>7.1f}ms\n"
summary += f"""----------------------------------------------------------
Effective Processing FPS : {avg_fps_val:.1f}
==========================================================
Report Assets saved to /content/report_assets/
  - sample_frames/output_frame_1..6.png
  - input_frames/input_frame_1..4.png
  - bev_roi_frame.png
  - yolo_only_frame.png
  - lane_only_frame.png
==========================================================
"""

with open("/content/pipeline_summary.txt", "w") as f:
    f.write(summary)
print("\n✅ Pipeline summary saved: /content/pipeline_summary.txt")
print(summary)

# ── 4. BEV Warped Frame (for report) ──────────────────────────────────────
sample_raw = cv2.imread("/content/report_assets/input_frames/input_frame_1.png")
if sample_raw is not None:
    h_r, w_r = sample_raw.shape[:2]
    src_bev = np.float32([
        [w_r * 0.15, h_r], [w_r * 0.85, h_r],
        [w_r * 0.62, h_r * 0.62], [w_r * 0.38, h_r * 0.62]
    ])
    dst_bev = np.float32([
        [w_r * 0.20, h_r], [w_r * 0.80, h_r],
        [w_r * 0.80, 0],   [w_r * 0.20, 0]
    ])
    H_mat = cv2.getPerspectiveTransform(src_bev, dst_bev)
    warped = cv2.warpPerspective(sample_raw, H_mat, (w_r, h_r))
    cv2.imwrite("/content/report_assets/bev_warped_output.png", warped)
    print("✅ BEV warped output saved: /content/report_assets/bev_warped_output.png")
else:
    print("⚠️ Input frame not found — run Cell 7 first")


## Cell 10 — Download ALL Files
> Downloads everything needed for the LaTeX report in one go.


In [ ]:
from google.colab import files
import os, glob

# ── Core output files ──────────────────────────────────────────────────────
core_files = [
    (OUTPUT_VIDEO,                             "processed video"),
    ("/content/latency_report.png",            "latency chart"),
    ("/content/latency_data.csv",              "latency summary CSV"),
    ("/content/latency_per_frame.csv",         "per-frame latency CSV"),
    ("/content/pipeline_summary.txt",          "pipeline summary"),
]

# ── Report asset images ────────────────────────────────────────────────────
image_files = [
    ("/content/report_assets/yolo_only_frame.png",  "YOLO-only frame"),
    ("/content/report_assets/lane_only_frame.png",  "Lane-only frame"),
    ("/content/report_assets/bev_roi_frame.png",    "BEV ROI frame"),
    ("/content/report_assets/bev_warped_output.png","BEV warped output"),
]

# Add all sample output frames
for p in sorted(glob.glob("/content/report_assets/sample_frames/output_frame_*.png")):
    image_files.append((p, f"output frame: {os.path.basename(p)}"))

# Add all input frames
for p in sorted(glob.glob("/content/report_assets/input_frames/input_frame_*.png")):
    image_files.append((p, f"input frame: {os.path.basename(p)}"))

all_files = core_files + image_files

print("📦 Downloading all report assets...\n")
downloaded, skipped = 0, 0
for fpath, label in all_files:
    if os.path.exists(fpath):
        print(f"  ⬇️  {label}: {os.path.basename(fpath)}")
        files.download(fpath)
        downloaded += 1
    else:
        print(f"  ⚠️  Skipped (not found): {fpath}")
        skipped += 1

print(f"\n✅ Done! {downloaded} files downloaded, {skipped} skipped.")
print()
print("📋 Rename these files and put them in your LaTeX images/ folder:")
print("   yolo_only_frame.png         → images/yolo_detection_example_1.png")
print("   lane_only_frame.png         → images/lane_detection_example_1.png")
print("   bev_roi_frame.png           → images/bev_roi_selection.png")
print("   bev_warped_output.png       → images/bev_warped_output.png")
print("   output_frame_1..6.png       → images/result_frame_1..6.png")
print("   input_frame_1..4.png        → images/input_video_frame_1..4.png")
print("   latency_report.png          → images/latency_graph.png")
print("   latency_data.csv            → paste values into LaTeX table")
print("   pipeline_summary.txt        → reference for report writing")
